In [1]:
import os
from dotenv import load_dotenv

# Change working directory to the parent folder (01_Langchain)
os.chdir(os.path.abspath(".."))
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


## Middleware

Middleware sits between the agent's core loop and its inputs/outputs, letting you intercept and modify behavior without changing the agent's underlying logic. Middleware is useful for the following:

- **Observability** — track agent behavior with logging, analytics, and debugging so you can see what the agent is doing and why.
- **Transformation** — rewrite prompts, filter or select which tools are available, and reformat outputs before they're returned.
- **Resilience** — add retries on failure, fallback models/tools, and early termination logic to stop runaway loops.
- **Safety** — enforce rate limits, guardrails, and PII detection/redaction to keep the agent's behavior within bounds.

### Summarization Middleware

As a conversation grows, the message history can exceed the model's context window or become expensive to send on every call. Summarization middleware solves this by automatically condensing older messages into a compact summary once a size threshold is hit, while keeping recent messages intact.

- **Trigger** — kicks in when the conversation history crosses a configured token/message limit.
- **Condense** — sends the older messages to an LLM to produce a short summary, replacing the originals.
- **Preserve recency** — keeps the most recent N messages verbatim so immediate context isn't lost.
- **Transparent to the agent** — the agent still sees a coherent history; it doesn't need to manage truncation itself.

In [ ]:
questions = [
    "What is 41 + 4?",
    "What is 2 - 9?",
    "What is 16 * 8?",
    "What is 9 + 4?",
    "What is 44 - 18?",
    "What is 6 * 19?",
    "What is 28 + 2?",
    "What is 2 - 3?",
    "What is 14 * 8?",
    "What is 33 + 20?",
    "What is 2 - 18?",
    "What is 13 * 18?",
    "What is 27 + 8?",
    "What is 29 - 19?",
    "What is 18 * 1?",
    "What is 49 + 6?",
    "What is 45 - 14?",
    "What is 22 * 9?",
    "What is 10 + 7?",
    "What is 49 - 11?",
    "What is 7 * 3?",
    "What is 25 + 4?",
    "What is 23 - 12?",
    "What is 39 * 9?",
    "What is 3 + 15?",
    "What is 35 - 4?",
    "What is 25 * 3?",
    "What is 36 + 10?",
    "What is 41 - 20?",
    "What is 24 * 19?",
    "What is 13 + 3?",
    "What is 3 - 8?",
    "What is 50 * 10?",
    "What is 6 + 8?",
    "What is 7 - 13?",
    "What is 18 * 15?",
    "What is 41 + 12?",
    "What is 11 - 12?",
    "What is 23 * 7?",
    "What is 43 + 9?",
    "What is 45 - 3?",
    "What is 39 * 6?",
    "What is 35 + 8?",
    "What is 11 - 15?",
    "What is 25 * 9?",
    "What is 41 + 18?",
    "What is 15 - 11?",
    "What is 50 * 2?",
    "What is 15 + 2?",
    "What is 21 - 13?",
    "What is 18 * 3?",
    "What is 14 + 19?",
    "What is 46 - 11?",
    "What is 14 * 16?",
    "What is 26 + 15?",
    "What is 10 - 9?",
    "What is 9 * 8?",
    "What is 48 + 18?",
    "What is 35 - 9?",
    "What is 48 * 19?",
    "What is 28 + 19?",
    "What is 26 - 12?",
    "What is 15 * 5?",
    "What is 33 + 16?",
    "What is 6 - 2?",
    "What is 8 * 5?",
    "What is 41 + 6?",
    "What is 44 - 14?",
    "What is 39 * 3?",
    "What is 25 + 13?",
    "What is 39 - 15?",
    "What is 34 * 9?",
    "What is 36 + 1?",
    "What is 44 - 4?",
    "What is 44 * 18?",
    "What is 49 + 9?",
    "What is 50 - 11?",
    "What is 8 * 10?",
]
len(questions)

### Model Fallback Middleware

If the primary model call fails (e.g. rate limit, timeout, outage), this middleware automatically retries the request with the next model in the fallback list, in order, until one succeeds.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[],
    middleware=[
        ModelFallbackMiddleware(
            "gpt-5.4-mini",
            "gpt-4o-mini",
        ),
    ],
)

In [ ]:
response = agent.invoke({"messages": [{"role": "user", "content": "What is 2+2?"}]})

last_message = response["messages"][-1]
print(last_message.content)
print(last_message.response_metadata.get("model_name"))